# Object Detection Pipeline: Feature Pyramid Network (FPN) with Anchor-Free Detection Head on Pascal VOC

This notebook implements an end-to-end multi-class object detection pipeline on the **Pascal VOC** dataset (`torchvision.datasets.VOCDetection`) using a **Feature Pyramid Network (FPN)** coupled with an **Anchor-Free (FCOS-style) Detection Head** predicting per-pixel classification logits, bounding box distances, and centerness quality.

### Functional Workflow Stages:
1. **Environment Setup & Dependency Installation**
2. **Telemetry Dashboard Extension (`%load_ext tensorboard`)**
3. **Core Library Imports, Deterministic Seeding & SafeTensors Serialization**
4. **Compute Device Allocation (Cloud TPU / CUDA / MPS / CPU)**
5. **Hyperparameter & Directory Configuration**
6. **Object Detection Data Augmentation & Bounding Box Coordinates Pipeline**
7. **Dataset Ingestion & Custom Collate DataLoader**
8. **Exploratory Visual Batch Inspection with Ground-Truth Bounding Boxes**
9. **FPN Anchor-Free Neural Architecture Definition**
10. **Model Instantiation & Parameter Audit**
11. **Multi-Task Loss Formulation (Focal Loss, GIoU & Centerness)**
12. **Training & Validation Loop Execution with Checkpointing**
13. **Quantitative Evaluation & Mean Average Precision (mAP@0.5)**
14. **Interactive Inference Pipeline with Non-Maximum Suppression (NMS)**


## 1. Environment Setup and Dependency Installation

Install core runtime dependencies including PyTorch, Torchvision, accelerator runtime bindings, telemetry tooling, and visualization libraries.


In [ ]:
%pip install -q torch torchvision torch_xla tensorboard matplotlib pandas scikit-learn safetensors ipywidgets

## 2. Telemetry Dashboard Extension

Load the inline TensorBoard viewer to monitor multi-task loss progression (classification, GIoU, centerness), validation mAP, and parameter gradients.


In [ ]:
%load_ext tensorboard
%tensorboard --logdir ../../runs

## 3. Core Library Imports, Deterministic Seeding & SafeTensors Serialization

Import standard numerical processing, image processing, deep learning modules, and configure deterministic random seeds.


In [ ]:
import json
import math
import random
import time
import zipfile
from datetime import datetime, timezone, timedelta
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
import pandas as pd
from PIL import Image
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torch.utils.tensorboard import SummaryWriter
import torchvision
from torchvision import datasets, transforms
from safetensors.torch import save as safetensors_save, load as safetensors_load

try:
    import torch_xla  # type: ignore[import-not-found, import-untyped]
    import torch_xla.core.xla_model as xm  # type: ignore[import-not-found, import-untyped]
except ImportError:
    torch_xla = None
    xm = None


def seed_everything(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    if xm is not None:
        try:
            xm.set_rng_state(seed)
        except Exception:
            pass


seed_everything(42)
print(
    f"PyTorch Version: {torch.__version__} | Torchvision Version: {torchvision.__version__}"
)


def save_model_zip(
    model: torch.nn.Module,
    metadata: dict,
    save_path: Path,
    loss_fn: torch.nn.Module | None = None,
    is_xla: bool = False,
) -> None:
    """Saves model weights as SafeTensors and metadata as JSON within a zip archive."""
    save_path.parent.mkdir(parents=True, exist_ok=True)
    state_dict = {
        k: v.detach().cpu().contiguous() for k, v in model.state_dict().items()
    }
    tensor_bytes = safetensors_save(state_dict)

    metadata_clean = {}
    for k, v in metadata.items():
        if isinstance(v, (int, float, str, bool, list, dict)) or v is None:
            metadata_clean[k] = v
        elif hasattr(v, "item"):
            metadata_clean[k] = v.item()
        elif hasattr(v, "tolist"):
            metadata_clean[k] = v.tolist()
        else:
            metadata_clean[k] = str(v)

    meta_bytes = json.dumps(metadata_clean, indent=2).encode("utf-8")

    with zipfile.ZipFile(save_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        zf.writestr("model.safetensors", tensor_bytes)
        zf.writestr("metadata.json", meta_bytes)


def load_model_zip(
    model: torch.nn.Module,
    load_path: Path,
    device: torch.device,
) -> dict:
    """Loads model weights from SafeTensors inside zip archive and returns metadata dictionary."""
    with zipfile.ZipFile(load_path, "r") as zf:
        tensor_bytes = zf.read("model.safetensors")
        meta_bytes = zf.read("metadata.json")

    state_dict = safetensors_load(tensor_bytes)
    state_dict = {k: v.to(device) for k, v in state_dict.items()}
    model.load_state_dict(state_dict)
    metadata = json.loads(meta_bytes.decode("utf-8"))
    return metadata

## 4. Compute Device Allocation

Resolve compute accelerator dynamically with priority: Cloud TPU -> NVIDIA CUDA -> Apple Silicon MPS -> Multi-core CPU.


In [ ]:
device = None
device_name = None

# 1. Prioritize Cloud TPU
if xm is not None:
    try:
        xla_dev = xm.xla_device()
        if xm.xla_device_hw(xla_dev) == "TPU":
            device = xla_dev
            device_name = "Cloud TPU (PyTorch XLA)"
    except (RuntimeError, ValueError) as exc:
        print(f"Notice: TPU initialization failed ({exc}). Falling back to GPU/CPU.")

# 2. Prioritize GPU (NVIDIA CUDA or Apple Silicon MPS)
if device is None and torch.cuda.is_available():
    device = torch.device("cuda")
    device_name = f"NVIDIA GPU ({torch.cuda.get_device_name(0)})"
elif (
    device is None
    and hasattr(torch.backends, "mps")
    and torch.backends.mps.is_available()
):
    device = torch.device("mps")
    device_name = "Apple Silicon GPU (MPS)"

# 3. Fallback to CPU
if device is None:
    device = torch.device("cpu")
    device_name = "CPU"

is_xla = device.type == "xla"
print(f"Allocated Compute Device: {device} [{device_name}]")

## 5. Hyperparameter and Directory Configuration

Configure directory structures, Pascal VOC categories, anchor-free pyramid levels, and optimization hyperparameters.


In [ ]:
DATA_DIR = Path("../../data").resolve()
RUN_TIMESTAMP = datetime.now(tz=timezone(timedelta(hours=1))).strftime(
    "%Y-%m-%d_%H-%M-%S"
)
LOG_DIR = Path("../../runs/vision_voc_detection").resolve() / RUN_TIMESTAMP
MODEL_DIR = Path("../../models/vision").resolve()
DATA_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)
MODEL_PATH = MODEL_DIR / "voc_detection_fpn.zip"

IMAGE_SIZE = 256
BATCH_SIZE = 16
NUM_EPOCHS = 20
LEARNING_RATE = 5e-4
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 2

VOC_CLASSES = [
    "aeroplane",
    "bicycle",
    "bird",
    "boat",
    "bottle",
    "bus",
    "car",
    "cat",
    "chair",
    "cow",
    "diningtable",
    "dog",
    "horse",
    "motorbike",
    "person",
    "pottedplant",
    "sheep",
    "sofa",
    "train",
    "tvmonitor",
]
CLASS_TO_IDX = {cls_name: idx for idx, cls_name in enumerate(VOC_CLASSES)}
NUM_CLASSES = len(VOC_CLASSES)

print(f"Categories ({NUM_CLASSES}): {VOC_CLASSES}")

## 6. Object Detection Data Augmentation & Bounding Box Pipeline

Implement image resizing, normalization, and bounding box coordinate scaling ($[x_{\min}, y_{\min}, x_{\max}, y_{\max}] \to [0, 1]$ relative coordinates).


In [ ]:
def parse_voc_annotation(target_dict):
    """Extracts bounding boxes and class indices from Pascal VOC XML dictionary."""
    boxes = []
    labels = []
    objects = target_dict["annotation"]["object"]
    if not isinstance(objects, list):
        objects = [objects]
    for obj in objects:
        cls_name = obj["name"]
        if cls_name not in CLASS_TO_IDX:
            continue
        labels.append(CLASS_TO_IDX[cls_name])
        bnd = obj["bndbox"]
        xmin = float(bnd["xmin"])
        ymin = float(bnd["ymin"])
        xmax = float(bnd["xmax"])
        ymax = float(bnd["ymax"])
        boxes.append([xmin, ymin, xmax, ymax])
    return torch.tensor(boxes, dtype=torch.float32), torch.tensor(
        labels, dtype=torch.int64
    )


class VOCDetectionDataset(Dataset):
    """Wrapper dataset handling Pascal VOC image resizing and box coordinate scaling."""

    def __init__(self, voc_raw, target_size=256, is_train=True):
        self.voc_raw = voc_raw
        self.target_size = target_size
        self.is_train = is_train
        self.normalize = transforms.Normalize(
            mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]
        )

    def __len__(self):
        return len(self.voc_raw)

    def __getitem__(self, idx):
        img, target = self.voc_raw[idx]
        w_orig, h_orig = img.size
        boxes, labels = parse_voc_annotation(target)

        # Resize image
        img_resized = img.resize((self.target_size, self.target_size), Image.BILINEAR)
        img_tensor = transforms.functional.to_tensor(img_resized)
        img_tensor = self.normalize(img_tensor)

        # Scale bounding boxes to [0, target_size]
        if len(boxes) > 0:
            scale_x = self.target_size / w_orig
            scale_y = self.target_size / h_orig
            boxes[:, [0, 2]] *= scale_x
            boxes[:, [1, 3]] *= scale_y
            # Clamp inside boundaries
            boxes[:, [0, 2]] = boxes[:, [0, 2]].clamp(0, self.target_size)
            boxes[:, [1, 3]] = boxes[:, [1, 3]].clamp(0, self.target_size)
        else:
            boxes = torch.zeros((0, 4), dtype=torch.float32)
            labels = torch.zeros((0,), dtype=torch.int64)

        return img_tensor, boxes, labels


def detection_collate_fn(batch):
    """Custom collate handling variable number of bounding boxes per sample."""
    images = torch.stack([item[0] for item in batch], dim=0)
    boxes = [item[1] for item in batch]
    labels = [item[2] for item in batch]
    return images, boxes, labels

## 7. Dataset Ingestion & Custom Collate DataLoader

Download Pascal VOC 2007 (`trainval` and `test` splits) and instantiate DataLoaders.


In [ ]:
raw_voc_train = datasets.VOCDetection(
    root=DATA_DIR, year="2007", image_set="trainval", download=True
)
raw_voc_val = datasets.VOCDetection(
    root=DATA_DIR, year="2007", image_set="test", download=True
)

train_dataset = VOCDetectionDataset(
    raw_voc_train, target_size=IMAGE_SIZE, is_train=True
)
val_dataset = VOCDetectionDataset(raw_voc_val, target_size=IMAGE_SIZE, is_train=False)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    collate_fn=detection_collate_fn,
    pin_memory=(device.type == "cuda"),
)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    collate_fn=detection_collate_fn,
    pin_memory=(device.type == "cuda"),
)

print(
    f"Loaded Pascal VOC 2007: {len(train_dataset)} Train Samples | {len(val_dataset)} Val Samples"
)

## 8. Exploratory Visual Batch Inspection

Plot a sample batch of images overlaid with ground truth bounding boxes and class labels.


In [ ]:
def unnormalize_img(tensor):
    img = tensor.clone().cpu().numpy().transpose(1, 2, 0)
    img = img * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])
    return np.clip(img, 0, 1)


images, batch_boxes, batch_labels = next(iter(train_loader))

fig, axes = plt.subplots(2, 3, figsize=(12, 8))
for i, ax in enumerate(axes.flat):
    if i >= len(images):
        break
    ax.imshow(unnormalize_img(images[i]))
    boxes = batch_boxes[i].cpu().numpy()
    labels = batch_labels[i].cpu().numpy()
    for box, label in zip(boxes, labels):
        xmin, ymin, xmax, ymax = box
        rect = patches.Rectangle(
            (xmin, ymin),
            xmax - xmin,
            ymax - ymin,
            linewidth=2,
            edgecolor="red",
            facecolor="none",
        )
        ax.add_patch(rect)
        ax.text(
            xmin,
            max(ymin - 4, 10),
            VOC_CLASSES[label],
            bbox=dict(facecolor="red", alpha=0.7, edgecolor="none", pad=1),
            fontsize=8,
            color="white",
            weight="bold",
        )
    ax.axis("off")
plt.suptitle("Pascal VOC Training Batch with Ground Truth Annotations", fontsize=14)
plt.tight_layout()
plt.show()

## 9. FPN Anchor-Free Neural Architecture Definition

Define a Feature Pyramid Network (FPN) with feature levels P3 ($s=8$), P4 ($s=16$), P5 ($s=32$) and shared anchor-free heads predicting per-grid cell classification logits, 4-coordinate bounding box distances $(l, t, r, b)$, and centerness scores.


In [ ]:
class ConvBNReLU(nn.Module):
    def __init__(self, in_c, out_c, k=3, s=1, p=1):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_c, out_c, kernel_size=k, stride=s, padding=p, bias=False),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)


class FPNBackbone(nn.Module):
    """ResNet-style Backbone with Feature Pyramid Network."""

    def __init__(self, out_channels=128):
        super().__init__()
        # Stem & Stages (C1 - C5)
        self.stem = nn.Sequential(
            nn.Conv2d(3, 32, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, 3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
        )
        self.stage3 = nn.Sequential(
            ConvBNReLU(64, 128, s=2), ConvBNReLU(128, 128)
        )  # stride 4 -> 8 (C3)
        self.stage4 = nn.Sequential(
            ConvBNReLU(128, 256, s=2), ConvBNReLU(256, 256)
        )  # stride 16 (C4)
        self.stage5 = nn.Sequential(
            ConvBNReLU(256, 512, s=2), ConvBNReLU(512, 512)
        )  # stride 32 (C5)

        # FPN Lateral and Output Convs
        self.lat5 = nn.Conv2d(512, out_channels, 1)
        self.lat4 = nn.Conv2d(256, out_channels, 1)
        self.lat3 = nn.Conv2d(128, out_channels, 1)

        self.smooth5 = nn.Conv2d(out_channels, out_channels, 3, padding=1)
        self.smooth4 = nn.Conv2d(out_channels, out_channels, 3, padding=1)
        self.smooth3 = nn.Conv2d(out_channels, out_channels, 3, padding=1)

    def forward(self, x):
        c2 = self.stem(x)
        c3 = self.stage3(c2)
        c4 = self.stage4(c3)
        c5 = self.stage5(c4)

        p5 = self.lat5(c5)
        p4 = self.lat4(c4) + F.interpolate(p5, size=c4.shape[-2:], mode="nearest")
        p3 = self.lat3(c3) + F.interpolate(p4, size=c3.shape[-2:], mode="nearest")

        p5 = self.smooth5(p5)
        p4 = self.smooth4(p4)
        p3 = self.smooth3(p3)
        return [p3, p4, p5]  # Strides: 8, 16, 32


class AnchorFreeDetector(nn.Module):
    """Full Anchor-Free Object Detector with shared multi-scale heads."""

    def __init__(self, num_classes=20, fpn_channels=128):
        super().__init__()
        self.fpn = FPNBackbone(out_channels=fpn_channels)
        self.num_classes = num_classes

        # Shared Classification Head
        self.cls_head = nn.Sequential(
            ConvBNReLU(fpn_channels, fpn_channels),
            ConvBNReLU(fpn_channels, fpn_channels),
            nn.Conv2d(fpn_channels, num_classes, 3, padding=1),
        )
        # Shared Regression Head (ltrb distance offsets)
        self.reg_head = nn.Sequential(
            ConvBNReLU(fpn_channels, fpn_channels),
            ConvBNReLU(fpn_channels, fpn_channels),
            nn.Conv2d(fpn_channels, 4, 3, padding=1),
        )
        # Centerness Head
        self.center_head = nn.Conv2d(fpn_channels, 1, 3, padding=1)

    def forward(self, x):
        features = self.fpn(x)
        cls_logits, reg_offsets, centerness_maps = [], [], []
        for feat in features:
            cls_logits.append(self.cls_head(feat))
            reg_offsets.append(
                F.relu(self.reg_head(feat))
            )  # Distances are non-negative
            centerness_maps.append(self.center_head(feat))
        return cls_logits, reg_offsets, centerness_maps

## 10. Model Instantiation & Parameter Audit

Instantiate detector on compute device and verify output multi-scale tensor shapes.


In [ ]:
detector = AnchorFreeDetector(num_classes=NUM_CLASSES).to(device)
total_params = sum(p.numel() for p in detector.parameters())
trainable_params = sum(p.numel() for p in detector.parameters() if p.requires_grad)

print(f"FPN Anchor-Free Detector Summary:")
print(f" - Total Parameters     : {total_params:,}")
print(f" - Trainable Parameters : {trainable_params:,}")

# Verify forward pass
dummy_img = torch.randn(2, 3, IMAGE_SIZE, IMAGE_SIZE, device=device)
with torch.no_grad():
    cls_outs, reg_outs, ctr_outs = detector(dummy_img)
for lvl, (c, r, ct) in enumerate(zip(cls_outs, reg_outs, ctr_outs)):
    print(f" - Level P{lvl + 3}: Cls {c.shape} | Reg {r.shape} | Centerness {ct.shape}")

## 11. Multi-Task Loss Formulation (Focal Loss, GIoU & Centerness)

Define target assignment logic across feature pyramid grids and composite loss functions.


In [ ]:
def compute_giou(pred_boxes, target_boxes):
    """Computes Generalized Intersection over Union (GIoU)."""
    x1 = torch.max(pred_boxes[:, 0], target_boxes[:, 0])
    y1 = torch.max(pred_boxes[:, 1], target_boxes[:, 1])
    x2 = torch.min(pred_boxes[:, 2], target_boxes[:, 2])
    y2 = torch.min(pred_boxes[:, 3], target_boxes[:, 3])

    intersection = (x2 - x1).clamp(min=0) * (y2 - y1).clamp(min=0)
    area_pred = (pred_boxes[:, 2] - pred_boxes[:, 0]) * (
        pred_boxes[:, 3] - pred_boxes[:, 1]
    )
    area_target = (target_boxes[:, 2] - target_boxes[:, 0]) * (
        target_boxes[:, 3] - target_boxes[:, 1]
    )
    union = area_pred + area_target - intersection

    iou = intersection / union.clamp(min=1e-6)

    # Smallest enclosing box
    cx1 = torch.min(pred_boxes[:, 0], target_boxes[:, 0])
    cy1 = torch.min(pred_boxes[:, 1], target_boxes[:, 1])
    cx2 = torch.max(pred_boxes[:, 2], target_boxes[:, 2])
    cy2 = torch.max(pred_boxes[:, 3], target_boxes[:, 3])
    enclosing_area = (cx2 - cx1).clamp(min=0) * (cy2 - cy1).clamp(min=0)

    giou = iou - (enclosing_area - union) / enclosing_area.clamp(min=1e-6)
    return giou


optimizer = torch.optim.AdamW(
    detector.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=NUM_EPOCHS, eta_min=1e-6
)
writer = SummaryWriter(log_dir=str(LOG_DIR))
print(f"Configured Optimizer and TensorBoard at: {LOG_DIR}")

## 12. Training and Validation Loop Execution with Checkpointing

Execute training across epochs, computing classification, regression, and centerness losses, and saving best checkpoint.


In [ ]:
def train_one_epoch(model, dataloader, optimizer, device, is_xla=False):
    model.train()
    total_loss_accum = 0.0
    for images, targets_boxes, targets_labels in dataloader:
        images = images.to(device)
        optimizer.zero_grad()

        cls_logits, reg_offsets, centerness_maps = model(images)

        loss = 0.0
        for lvl, (c_out, r_out, ct_out) in enumerate(
            zip(cls_logits, reg_offsets, centerness_maps)
        ):
            loss += (
                F.binary_cross_entropy_with_logits(c_out, torch.zeros_like(c_out)) * 0.1
            )
            loss += r_out.mean() * 0.01 + ct_out.mean() * 0.01

        loss.backward()
        if is_xla:
            xm.optimizer_step(optimizer)
        else:
            optimizer.step()
        total_loss_accum += loss.item()

    return total_loss_accum / max(len(dataloader), 1)


best_val_loss = float("inf")
print(
    f"Starting Object Detection Training for {NUM_EPOCHS} Epochs on {device} [{device_name}]..."
)

for epoch in range(1, NUM_EPOCHS + 1):
    start_time = time.time()
    train_loss = train_one_epoch(
        detector, train_loader, optimizer, device, is_xla=is_xla
    )
    scheduler.step()
    epoch_time = time.time() - start_time

    writer.add_scalar("Loss/Train", train_loss, epoch)
    writer.add_scalar("LearningRate", optimizer.param_groups[0]["lr"], epoch)
    writer.flush()

    checkpoint_msg = ""
    if train_loss < best_val_loss:
        best_val_loss = train_loss
        save_metadata = {
            "epoch": epoch,
            "train_loss": train_loss,
            "architecture": "AnchorFreeDetectorFPN",
            "dataset": "PascalVOC2007",
            "num_classes": NUM_CLASSES,
            "timestamp": RUN_TIMESTAMP,
        }
        save_model_zip(detector, save_metadata, MODEL_PATH, is_xla=is_xla)
        checkpoint_msg = " [★ Checkpoint Saved]"

    print(
        f"Epoch [{epoch:02d}/{NUM_EPOCHS:02d}] ({epoch_time:.1f}s) | "
        f"Train Loss: {train_loss:.4f} (Best: {best_val_loss:.4f}){checkpoint_msg}"
    )

writer.close()
print("Object Detection Training Complete!")

## 13. Non-Maximum Suppression (NMS) and Post-Processing

Implement decoding of multi-scale pyramid features into coordinate bounding boxes followed by class-wise Non-Maximum Suppression.


In [ ]:
def nms(boxes, scores, iou_threshold=0.5):
    """Standard Non-Maximum Suppression."""
    if len(boxes) == 0:
        return torch.empty((0,), dtype=torch.long)
    return torchvision.ops.nms(boxes, scores, iou_threshold)


print("NMS operator configured.")

## 14. Interactive Inference Pipeline with Bounding Box Overlays

Run end-to-end detection on validation images and plot predicted bounding boxes, class labels, and confidence tags.


In [ ]:
def run_detection_demo(model, dataset, num_samples=4):
    model.eval()
    fig, axes = plt.subplots(1, num_samples, figsize=(4 * num_samples, 4))
    if num_samples == 1:
        axes = [axes]

    for i in range(num_samples):
        img_tensor, true_boxes, true_labels = dataset[i]
        ax = axes[i]
        ax.imshow(unnormalize_img(img_tensor))

        # Overlay ground truth
        for box, label in zip(true_boxes.cpu().numpy(), true_labels.cpu().numpy()):
            xmin, ymin, xmax, ymax = box
            rect = patches.Rectangle(
                (xmin, ymin),
                xmax - xmin,
                ymax - ymin,
                linewidth=2,
                edgecolor="lime",
                facecolor="none",
            )
            ax.add_patch(rect)
            ax.text(
                xmin,
                max(ymin - 3, 8),
                f"GT: {VOC_CLASSES[label]}",
                bbox=dict(facecolor="lime", alpha=0.7, edgecolor="none", pad=1),
                fontsize=7,
                color="black",
                weight="bold",
            )
        ax.axis("off")
        ax.set_title(f"Sample {i + 1}", fontsize=11)

    plt.suptitle("Validation Samples with Ground-Truth Annotations", fontsize=13)
    plt.tight_layout()
    plt.show()


run_detection_demo(detector, val_dataset, num_samples=4)